# lineup on Kaggle

**Before running:** open **Settings** (right panel) -> **Accelerator -> GPU T4 x2**, and set **Internet -> On** (needed to clone the repo and download the models). Then **Run All**.

Kaggle gives ~30 GPU-hours per week and 12-hour sessions, so a few-hundred-case run finishes comfortably. Work is saved after each stage, so a timeout never loses finished cases. Results land in /kaggle/working -- download `lineup_study.zip` from the Output panel when it's done.

In [ ]:
import os, sys
if not os.path.isdir("/kaggle/working/LINEUP"):
    !git clone -q https://github.com/santoshcheethiralame-dot/LINEUP /kaggle/working/LINEUP
%cd /kaggle/working/LINEUP
!pip install -q bitsandbytes
!pip install -q -e . --no-deps
if "/kaggle/working/LINEUP/src" not in sys.path:
    sys.path.insert(0, "/kaggle/working/LINEUP/src")
import lineup, torch
print("lineup", lineup.__version__, "| cuda:", torch.cuda.is_available())

## Settings

Start with `LIMIT = 300` to get a result fast; raise it for tighter confidence intervals. The models run in 4-bit on a single T4. Lower `N_ABLATIONS` if you want it faster.

In [ ]:
LIMIT = 300
K = 6
SPLIT = "validation"
DATASET = "hotpotqa"   # or "2wiki"
SEED = 0
MAX_NEW_TOKENS = 32
N_ABLATIONS = 16       # ContextCite ablations per case; lower = faster, noisier
HARD_TRAPS = False     # True adds a redundant decoy per case (the "harder traps" experiment)
METHODS_SCOPE = "wrong"  # "wrong" runs methods only on wrong cases (~2x faster); "all" also scores the abstention margins

MODELS = {
    "qwen": "Qwen/Qwen2.5-7B-Instruct",
    "phi": "microsoft/Phi-3.5-mini-instruct",
}

In [ ]:
from pathlib import Path

import torch
from tqdm.auto import tqdm

from lineup.backends import TransformersModel
from lineup.config import set_seed
from lineup.correctness import LLMJudge
from lineup.data.scenario import ScenarioBuilder
from lineup.data.schema import CaseRoles
from lineup.data.serialization import write_generations, write_predictions, write_roles, write_scenarios
from lineup.data.sources import load_examples
from lineup.data.substitution import build_answer_pool
from lineup.generation import generate_and_judge
from lineup.methods import ContextCite, LexicalSimilarity, LLMJudgeCulprit, SingleChunkSupport, run_method
from lineup.oracle import leave_one_out

set_seed(SEED)
examples = list(load_examples(DATASET, SPLIT, limit=LIMIT))
builder = ScenarioBuilder(answer_pool=build_answer_pool(examples), k=K, seed=SEED, hard_traps=HARD_TRAPS)
scenarios = [s for s in (builder.build(e) for e in examples) if s is not None]
decoys = sum(1 for s in scenarios for c in s.chunks if c.provenance == "decoy")
print(f"built {len(scenarios)} cases from {len(examples)} questions | hard_traps={HARD_TRAPS} | decoy chunks={decoys}")

for name, model_id in MODELS.items():
    out = Path("runs") / name
    out.mkdir(parents=True, exist_ok=True)
    write_scenarios(out / "scenarios.jsonl", scenarios)
    set_seed(SEED)
    model = TransformersModel(model_id, max_new_tokens=MAX_NEW_TOKENS, load_in_4bit=True)
    judge = LLMJudge(model)

    # Save after each stage, so a session timeout never throws away finished work.
    generations = [generate_and_judge(model, s, llm_judge=judge) for s in tqdm(scenarios, desc=f"{name}: generate")]
    write_generations(out / "generations.jsonl", generations)

    role_cases = []
    for s, g in tqdm(list(zip(scenarios, generations)), desc=f"{name}: oracle"):
        if g.is_correct:
            role_cases.append(CaseRoles(s.qid, s.question, s.gold_answer, g.model_answer, True, []))
        else:
            role_cases.append(leave_one_out(model, s, g, llm_judge=judge))
    write_roles(out / "roles.jsonl", role_cases)

    methods = [ContextCite(n_ablations=N_ABLATIONS, seed=SEED), LexicalSimilarity(), LLMJudgeCulprit(), SingleChunkSupport()]
    method_cases = [(s, g) for s, g in zip(scenarios, generations) if METHODS_SCOPE == "all" or not g.is_correct]
    predictions = []
    for s, g in tqdm(method_cases, desc=f"{name}: methods"):
        for m in methods:
            predictions.append(run_method(m, model, s, g.model_answer))
    write_predictions(out / "predictions.jsonl", predictions)
    print(f"{name}: {sum(not g.is_correct for g in generations)} wrong of {len(scenarios)} -> {out}")

    del model, judge
    torch.cuda.empty_cache()

## Results

Four panels: misleading-as-culprit rate with 95% bootstrap intervals, the predicted-role heatmap, the selective-answering risk-coverage curves, and the cross-model agreement.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

from lineup.agreement import compare_models
from lineup.data.serialization import read_generations, read_predictions, read_roles
from lineup.downstream import abstention_curves
from lineup.scoring import bootstrap_intervals, score_predictions

ROLE_ORDER = ["culprit", "misleading", "silent", "inert"]


def render_dashboard(run_dirs, seed=0, n_boot=2000):
    data = {}
    for name, directory in run_dirs.items():
        base = Path(directory)
        roles = read_roles(base / "roles.jsonl")
        wrong = [case for case in roles if not case.original_correct]
        preds = read_predictions(base / "predictions.jsonl")
        data[name] = {
            "roles": roles, "wrong": wrong, "preds": preds,
            "gens": read_generations(base / "generations.jsonl"),
            "reports": {r.method: r for r in score_predictions(wrong, preds)},
            "cis": bootstrap_intervals(wrong, preds, n_boot=n_boot, seed=seed),
        }

    methods = sorted(next(iter(data.values()))["reports"])
    names = list(run_dirs)
    primary = names[0]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))

    ax = axes[0, 0]
    width = 0.8 / len(names)
    for i, name in enumerate(names):
        reps, cis = data[name]["reports"], data[name]["cis"]
        vals = [reps[m].misleading_as_culprit_rate for m in methods]
        lo, hi = [], []
        for j, m in enumerate(methods):
            low, high = cis[m]["misleading_as_culprit_rate"]
            lo.append(vals[j] - (low if low is not None else vals[j]))
            hi.append((high if high is not None else vals[j]) - vals[j])
        ax.bar([x + i * width for x in range(len(methods))], vals, width, yerr=[lo, hi], capsize=4, label=name)
    ax.set_xticks([x + width * (len(names) - 1) / 2 for x in range(len(methods))])
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel("misleading-as-culprit rate")
    ax.set_title("How often each method blames the near-miss")
    ax.set_ylim(0, 1)
    ax.legend()

    ax = axes[0, 1]
    reps = data[primary]["reports"]
    matrix = [[reps[m].predicted_role_rate.get(r, 0.0) for r in ROLE_ORDER] for m in methods]
    image = ax.imshow(matrix, cmap="magma", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(ROLE_ORDER)))
    ax.set_xticklabels(ROLE_ORDER)
    ax.set_yticks(range(len(methods)))
    ax.set_yticklabels(methods)
    for r in range(len(methods)):
        for c in range(len(ROLE_ORDER)):
            ax.text(c, r, f"{matrix[r][c]:.2f}", ha="center", va="center",
                    color="white" if matrix[r][c] < 0.5 else "black", fontsize=9)
    ax.set_title(f"Where {primary}'s predicted culprit truly lands")
    fig.colorbar(image, ax=ax, fraction=0.046)

    ax = axes[1, 0]
    for signal, (cov, risk) in abstention_curves(data[primary]["gens"], data[primary]["preds"], data[primary]["roles"]).items():
        if cov:
            ax.plot(cov, risk, label=signal, linewidth=1.5)
    ax.set_xlabel("coverage")
    ax.set_ylabel("risk (error of the answered set)")
    ax.set_title("Selective answering -- does attribution help abstain")
    ax.legend(fontsize=8)

    ax = axes[1, 1]
    ax.axis("off")

    def fmt(value):
        return f"{value:.2f}" if value is not None else "n/a"

    if len(names) >= 2:
        rep = compare_models(data[names[0]]["roles"], data[names[1]]["roles"])
        lines = [
            f"cross-model agreement: {names[0]} vs {names[1]}", "",
            f"cases wrong in both:     {rep.n_both_wrong} / {rep.n_common}",
            f"same culprit set:        {fmt(rep.same_culprit_rate)}",
            f"culprit-set Jaccard:     {fmt(rep.culprit_jaccard)}",
            f"per-passage role kappa:  {fmt(rep.role_kappa)}",
        ]
        ax.text(0.0, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=12)
    else:
        ax.text(0.0, 0.95, "add a second model to fill this panel", va="top", fontsize=12)

    fig.tight_layout()
    fig.savefig("runs/dashboard.png", dpi=130, bbox_inches="tight")
    plt.show()

    for name in names:
        print(f"\n== {name} ==")
        for m in methods:
            report = data[name]["reports"][m]
            low, high = data[name]["cis"][m]["misleading_as_culprit_rate"]
            band = f"  [{low:.2f}, {high:.2f}]" if low is not None else ""
            top1 = f"{report.top1_culprit_accuracy:.2f}" if report.top1_culprit_accuracy is not None else "n/a"
            print(f"  {m:18s} top1={top1}  misleading-as-culprit={report.misleading_as_culprit_rate:.2f}{band}")


render_dashboard({name: f"runs/{name}" for name in MODELS}, seed=SEED, n_boot=2000)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/lineup_study", "zip", "runs")
print("done -- download lineup_study.zip from the Output panel on the right (it has the JSONL + dashboard.png)")